In [65]:
import mne
import io
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import pandas as pd
from glob import glob
import datetime
import time

# Compute TF means

In [26]:
arr_path = "/Users/fzaki001/Downloads/tf_arr/"
measures = [
    # "TF",
    # "ITPS",
    "ICPS",
    # "wPLI",
]

valid_id = dict()
condition = ["resp_s_i_0", "resp_s_i_1",
            # "resp_s_c_1",
            "resp_ns_i_0", "resp_ns_i_1",
            # "resp_ns_c_1"
            ]
for m in measures:
    for c in condition:
        file = glob(f"{arr_path}*{m}*{c}*.mat")[0]
        data = scipy.io.loadmat(file)
        valid_id[c] = list(data["subjects"][0])

subs_per_condition = []
for key in list(valid_id.keys()):
    print(len(valid_id[key]))
    subs_per_condition.append(valid_id[key])

134
137
132
137


In [28]:
subjects_with_all_conds = list(set.intersection(*[set(l) for l in subs_per_condition]))

In [30]:
m = "ICPS"
condition = ["resp_s_i_0", "resp_s_i_1",
            # "resp_s_c_1",
            "resp_ns_i_0", "resp_ns_i_1",
            # "resp_ns_c_1"
            ]

cond_idx = dict()
for c in condition:  
    file = glob(f"{arr_path}*{m}*{c}*.mat")[0]
    data = scipy.io.loadmat(file)
    cond_subs = list(data["subjects"][0])
    idx = []
    for sub in subjects_with_all_conds:
        idx.append(cond_subs.index(sub)+1) # this will have 1-based indices suitable for matlab
    cond_idx[c] = sorted(idx)
    # valid_id[c] = list(data["subjects"][0])

In [33]:
for c in condition:
    scipy.io.savemat(f"{arr_path}/idx_{c}.mat",
                         {f"sub_idx": np.array(cond_idx[c]),})

In [32]:
cond_idx

{'resp_s_i_0': [1,
  2,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  20,
  21,
  22,
  23,
  24,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  96,
  97,
  98,
  99,
  100,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  111,
  112,
  113,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134],
 'resp_s_i_1': [1,
  2,
  4,
  5,
  6,
  7,
  8,
  9,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  21,
  22,
  23,
  24,
  25,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
 

In [131]:
glob("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/TF_outputs/main/sub-*.mat")[0]

'/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/TF_outputs/main/sub-3000002_all_eeg_processed_data_s1_r1_e1_tf_TF_baselinecorrected_conditionresp_ns_c_1.mat'

In [126]:
with h5py.File(glob("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/TF_outputs/main/sub-*.mat")[0], 'r') as f:
    # Print the type of references to understand what we're dealing with
    refs = f['channel_location']['labels'][()]
    print("Type of refs:", type(refs))
    print("Shape of refs:", refs.shape)
    
    # Try to inspect a single reference
    single_ref = refs.flatten()
    print("Type of single ref:", type(single_ref))
    
    # Try to access the data
    try:
        data = [f[single_ref][()] for single_ref in refs.flatten()]
        print("Successfully accessed data")
    except Exception as e:
        print("Error accessing data:", str(e))

Type of refs: <class 'numpy.ndarray'>
Shape of refs: (64, 1)
Type of single ref: <class 'numpy.ndarray'>
Error accessing data: Accessing a group is done with bytes or str, not <class 'numpy.ndarray'>


In [6]:
import h5py
arr_path = "/Users/fzaki001/Downloads/tf_arr/"
helper_data = h5py.File(
    glob("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/TF_outputs/main/sub-*.mat")[0]
)

freqs = helper_data['frequency'][:]
times = helper_data['ds_time'][:]
ch_locs = [str(i) for i in range(1, 65)]

In [333]:
# NOT DIFFERENCE TF + ITPS

thrive_data = pd.read_csv("/Users/fzaki001/Downloads/thrive/derivatives/processed/summary.csv")["sub"].to_frame()

ch = ['1', '2', '33', '34']
measures = [
    "TF",
    "ITPS"
]
conditions = [
    "resp_s_i_0",
    "resp_s_i_1",
    "resp_s_c_1",
    "resp_ns_i_0",
    "resp_ns_i_1",
    "resp_ns_c_1",
            ]

for m in measures:
    for c in conditions:
        for band in ["theta", "delta"]:
            for window in ["early", "late"]:
                if band == "theta":
                    fmin = 4
                    fmax = 7
                elif band == "delta":
                    fmin = 1
                    fmax = 3

                if window == "early":
                    tmin = 0
                    tmax = 250
                elif window == "late":
                    tmin = 256
                    tmax = 504
                
                fmin_idx = np.argmin(np.abs(freqs-fmin))
                # assert freqs[fmin_idx] == fmin, "Check your freqs!"
                fmax_idx = np.argmin(np.abs(freqs-fmax))
                # assert freqs[fmax_idx] == fmax, "Check your freqs!"
                
                tmin_idx = np.argmin(np.abs(times-tmin))
                # assert times[tmin_idx] == tmin, "Check your times!"
                tmax_idx = np.argmin(np.abs(times-tmax))
                # assert times[tmax_idx] == tmax, "Check your times!"
                
                ch_idx = []
                for channel in ch:
                    if channel in ch_locs:
                        ch_idx.append(ch_locs.index(channel))
                
                # sub_idx = scipy.io.loadmat(f"{arr_path}/idx_{c}.mat")["sub_idx"][0]-1 # make it 0-based again
                tf_df = pd.DataFrame(columns = ["sub", f"{m}_{c}_{band}_{window}"])
                tf_arr = scipy.io.loadmat(f"{arr_path}/{m}_{c}.mat")
                sub_ids = tf_arr['subjects'][0]
                tf_data = tf_arr[f"{m}_{c}"]
                assert tf_data.shape[1:] == (64, 375, 59), f"Check your {m} data!"
                
                # for sub_id in sub_idx:
                for sub_id in range(tf_data.shape[0]):
                    # sub_avg = np.mean(tf_data[sub_id, :, :, :], 0)
                    sub_avg = tf_data[sub_id, :, :, :]
                    assert sub_avg.shape == (64, 375, 59), f"Check your {m} data!"
                    
                    ch_avg = np.mean(sub_avg[ch_idx, :, :], 0)
                    assert ch_avg.shape == (375, 59), f"Check your {m} data!"
                    
                    time_avg = np.mean(ch_avg[tmin_idx:tmax_idx+1, :], 0)
                    assert len(time_avg) == 59 and time_avg.ndim == 1, f"Check your {m} data!"
                    freq_avg = np.mean(time_avg[fmin_idx:fmax_idx+1], 0)
                
                    tf_df.loc[sub_id, "sub"] = sub_ids[sub_id]
                    tf_df.loc[sub_id, f"{m}_{c}_{band}_{window}"] = freq_avg
                
                thrive_data = thrive_data.merge(tf_df, on="sub", how="left")

In [336]:
# NOT DIFFERENCE ICPS

thrive_data = pd.read_csv("/Users/fzaki001/Downloads/thrive/derivatives/processed/summary.csv")["sub"].to_frame()

measures = [
    "ICPS",
]
conditions = [
    "resp_s_i_0",
    "resp_s_i_1",
    "resp_s_c_1",
    "resp_ns_i_0",
    "resp_ns_i_1",
    "resp_ns_c_1",
            ]

for m in measures:
    for c in conditions:
        for band in ["theta", "delta"]:
            for window in ["early", "late"]:
                for cluster in ["DLPFC_L", "DLPFC_R", "OCC_L", "OCC_R"]:
                    if cluster == "DLPFC_L":
                        ch = ['4', '6']
                    elif cluster == "DLPFC_R":
                        ch = ['36', '39']
                    if cluster == "OCC_L":
                        ch = ['22', '24']
                    elif cluster == "OCC_R":
                        ch = ['53', '55']
                        
                    if band == "theta":
                        fmin = 4
                        fmax = 7
                    elif band == "delta":
                        fmin = 1
                        fmax = 3
    
                    if window == "early":
                        tmin = 0
                        tmax = 250
                    elif window == "late":
                        tmin = 256
                        tmax = 504
                        
                    fmin_idx = np.argmin(np.abs(freqs-fmin))
                    # assert freqs[fmin_idx] == fmin, "Check your freqs!"
                    fmax_idx = np.argmin(np.abs(freqs-fmax))
                    # assert freqs[fmax_idx] == fmax, "Check your freqs!"
                    
                    tmin_idx = np.argmin(np.abs(times-tmin))
                    # assert times[tmin_idx] == tmin, "Check your times!"
                    tmax_idx = np.argmin(np.abs(times-tmax))
                    # assert times[tmax_idx] == tmax, "Check your times!"
                    
                    ch_idx = []
                    for channel in ch:
                        if channel in ch_locs:
                            ch_idx.append(ch_locs.index(channel))
                    
                    # sub_idx = scipy.io.loadmat(f"{arr_path}/idx_{c}.mat")["sub_idx"][0]-1 # make it 0-based again
                    tf_df = pd.DataFrame(columns = ["sub", f"{m}_{c}_{band}_{window}_{cluster}"])
                    tf_arr = scipy.io.loadmat(f"{arr_path}/{m}_{c}.mat")
                    sub_ids = tf_arr['subjects'][0]
                    tf_data = tf_arr[f"{m}_{c}"]
                    assert tf_data.shape[1:] == (64, 375, 59), f"Check your {m} data!"
                    
                    # for sub_id in sub_idx:
                    for sub_id in range(tf_data.shape[0]):
                        # sub_avg = np.mean(tf_data[sub_id, :, :, :], 0)
                        sub_avg = tf_data[sub_id, :, :, :]
                        assert sub_avg.shape == (64, 375, 59), f"Check your {m} data!"
                        
                        ch_avg = np.mean(sub_avg[ch_idx, :, :], 0)
                        assert ch_avg.shape == (375, 59), f"Check your {m} data!"
                        
                        time_avg = np.mean(ch_avg[tmin_idx:tmax_idx+1, :], 0)
                        assert len(time_avg) == 59 and time_avg.ndim == 1, f"Check your {m} data!"
                        freq_avg = np.mean(time_avg[fmin_idx:fmax_idx+1], 0)
                    
                        tf_df.loc[sub_id, "sub"] = sub_ids[sub_id]
                        tf_df.loc[sub_id, f"{m}_{c}_{band}_{window}_{cluster}"] = freq_avg
                    
                    thrive_data = thrive_data.merge(tf_df, on="sub", how="left")

In [113]:
freqs[fmin_idx:fmax_idx+1]

array([[4. ],
       [4.5],
       [5. ],
       [5.5],
       [6. ],
       [6.5],
       [7. ]])

In [49]:
# SOCIAL MINUS NON SOCIAL DIFFERENCE SCORES

thrive_data = pd.read_csv("/Users/fzaki001/Downloads/thrive/derivatives/processed/summary.csv")["sub"].to_frame()

ch = ['1', '2', '33', '34']
m = "TF"
c = [
    "resp_s_i_0",
    "resp_s_i_1",
    "resp_ns_i_0",
    "resp_ns_i_1",
            ]

# for m in measures:
    # for c in conditions:
        # for band in ["theta", "delta"]:
            # for window in ["early", "late"]:
for m in ["TF", "ITPS"]:
    band = 'theta'
    window = 'early'
    if band == "theta":
        fmin = 4
        fmax = 7
    elif band == "delta":
        fmin = 1
        fmax = 3
    
    if window == "early":
        tmin = 0
        tmax = 250
    elif window == "late":
        tmin = 256
        tmax = 504
    
    fmin_idx = np.argmin(np.abs(freqs-fmin))
    # assert freqs[fmin_idx] == fmin, "Check your freqs!"
    fmax_idx = np.argmin(np.abs(freqs-fmax))
    # assert freqs[fmax_idx] == fmax, "Check your freqs!"
    
    tmin_idx = np.argmin(np.abs(times-tmin))
    # assert times[tmin_idx] == tmin, "Check your times!"
    tmax_idx = np.argmin(np.abs(times-tmax))
    # assert times[tmax_idx] == tmax, "Check your times!"
    
    ch_idx = []
    for channel in ch:
        if channel in ch_locs:
            ch_idx.append(ch_locs.index(channel))
    
    # sub_idx = scipy.io.loadmat(f"{arr_path}/idx_{c}.mat")["sub_idx"][0]-1 # make it 0-based again
    tf_df = pd.DataFrame(
        columns = [
        "sub",
        f"{m}_{c[0]}_diff_{c[1]}_{band}_{window}",
        f"{m}_{c[2]}_diff_{c[3]}_{band}_{window}",
        f"{m}_s_diff_ns_{band}_{window}",
    ]
    )
    
    tf_arr_c1= scipy.io.loadmat(f"{arr_path}/{m}_{c[0]}.mat")
    tf_arr_c2= scipy.io.loadmat(f"{arr_path}/{m}_{c[1]}.mat")
    tf_arr_c3= scipy.io.loadmat(f"{arr_path}/{m}_{c[2]}.mat")
    tf_arr_c4= scipy.io.loadmat(f"{arr_path}/{m}_{c[3]}.mat")
    
    sub_ids_c1 = tf_arr_c1['subjects'][0]
    sub_ids_c2 = tf_arr_c2['subjects'][0]
    sub_ids_c3 = tf_arr_c3['subjects'][0]
    sub_ids_c4 = tf_arr_c4['subjects'][0]
    
    tf_data_c1 = tf_arr_c1[f"{m}_{c[0]}"]
    tf_data_c2 = tf_arr_c2[f"{m}_{c[1]}"]
    tf_data_c3 = tf_arr_c3[f"{m}_{c[2]}"]
    tf_data_c4 = tf_arr_c4[f"{m}_{c[3]}"]
    
    assert tf_data_c1.shape[1:] == (64, 375, 59), f"Check your {m} data!"
    assert tf_data_c2.shape[1:] == (64, 375, 59), f"Check your {m} data!"
    assert tf_data_c3.shape[1:] == (64, 375, 59), f"Check your {m} data!"
    assert tf_data_c4.shape[1:] == (64, 375, 59), f"Check your {m} data!"
    
    # for sub_id in sub_idx:
    for sub_id in range(tf_data_c1.shape[0]):
        # sub_avg = np.mean(tf_data[sub_id, :, :, :], 0)
        sub_avg_c1 = tf_data_c1[sub_id, :, :, :]
        assert sub_avg_c1.shape == (64, 375, 59), f"Check your {m} data!"
        
        sub_to_match_c2 = list(sub_ids_c2).index(sub_ids_c1[sub_id])
        sub_avg_c2 = tf_data_c2[sub_to_match_c2, :, :, :]
        assert sub_avg_c2.shape == (64, 375, 59), f"Check your {m} data!"
        
        assert(sub_ids_c1[sub_id] == sub_ids_c2[sub_to_match_c2]), "Your IDs are messed up!"
        
        sub_avg_d1 = np.subtract(sub_avg_c1, sub_avg_c2)  # s err minus s corr
        assert sub_avg_d1.shape == (64, 375, 59), f"Check your {m} data!"
        
        ch_avg_d1 = np.mean(sub_avg_d1[ch_idx, :, :], 0)
        assert ch_avg_d1.shape == (375, 59), f"Check your {m} data!"
        
        time_avg_d1 = np.mean(ch_avg_d1[tmin_idx:tmax_idx+1, :], 0)
        assert len(time_avg_d1) == 59 and time_avg_d1.ndim == 1, f"Check your {m} data!"
        
        freq_avg_d1 = np.mean(time_avg_d1[fmin_idx:fmax_idx+1], 0)
    
        tf_df.loc[sub_id, "sub"] = sub_ids_c1[sub_id]
        tf_df.loc[sub_id, f"{m}_{c[0]}_diff_{c[1]}_{band}_{window}"] = freq_avg_d1
    
        try:
            sub_to_match_c3 = list(sub_ids_c3).index(sub_ids_c1[sub_id])
            sub_to_match_c4 = list(sub_ids_c4).index(sub_ids_c1[sub_id])
        
            sub_avg_c3 = tf_data_c3[sub_to_match_c3, :, :, :]
            sub_avg_c4 = tf_data_c4[sub_to_match_c4, :, :, :]
        
            assert(int(sub_ids_c1[sub_id]) == int(sub_ids_c2[sub_to_match_c2])\
                   == int(sub_ids_c3[sub_to_match_c3]) == int(sub_ids_c4[sub_to_match_c4])), "Your IDs are messed up!"
        
            sub_avg_d2 = np.subtract(sub_avg_c3, sub_avg_c4) # ns err minus ns corr
            assert sub_avg_d2.shape == (64, 375, 59), f"Check your {m} data!"
            
            ch_avg_d2 = np.mean(sub_avg_d2[ch_idx, :, :], 0)
            assert ch_avg_d2.shape == (375, 59), f"Check your {m} data!"
            
            time_avg_d2 = np.mean(ch_avg_d2[tmin_idx:tmax_idx+1, :], 0)
            assert len(time_avg_d2) == 59 and time_avg_d2.ndim == 1, f"Check your {m} data!"
            
            freq_avg_d2 = np.mean(time_avg_d2[fmin_idx:fmax_idx+1], 0)
        
            # tf_df.loc[sub_id, "sub"] = sub_ids_c1[sub_id]
            tf_df.loc[sub_id, f"{m}_{c[2]}_diff_{c[3]}_{band}_{window}"] = freq_avg_d2
        
            sub_avg_d3 = np.subtract(sub_avg_d1, sub_avg_d2) # social minus nonsocial
            assert sub_avg_d3.shape == (64, 375, 59), f"Check your {m} data!"
            
            ch_avg_d3 = np.mean(sub_avg_d3[ch_idx, :, :], 0)
            assert ch_avg_d3.shape == (375, 59), f"Check your {m} data!"
            
            time_avg_d3 = np.mean(ch_avg_d3[tmin_idx:tmax_idx+1, :], 0)
            assert len(time_avg_d3) == 59 and time_avg_d3.ndim == 1, f"Check your {m} data!"
            
            freq_avg_d3 = np.mean(time_avg_d3[fmin_idx:fmax_idx+1], 0)
        
            # tf_df.loc[sub_id, "sub"] = sub_ids_c1[sub_id]
            tf_df.loc[sub_id, f"{m}_s_diff_ns_{band}_{window}"] = freq_avg_d3
        except:
            tf_df.loc[sub_id, f"{m}_{c[2]}_diff_{c[3]}_{band}_{window}"] = np.nan
            tf_df.loc[sub_id, f"{m}_s_diff_ns_{band}_{window}"] = np.nan

    thrive_data = thrive_data.merge(tf_df, on="sub", how="left")

In [53]:
# DIFFERENCE SCORES TF+ITPS

thrive_data = pd.read_csv("/Users/fzaki001/Downloads/thrive/derivatives/processed/summary.csv")["sub"].to_frame()

ch = ['1', '2', '33', '34']
# m = "TF"
conditions = [
    ["resp_s_i_0",
    "resp_s_i_1",],
    ["resp_ns_i_0",
    "resp_ns_i_1",],
            ]

# for m in measures:
    # for c in conditions:
        # for band in ["theta", "delta"]:
            # for window in ["early", "late"]:
for m in ["TF", "ITPS"]:
    band = 'theta'
    window = 'early'
    if band == "theta":
        fmin = 4
        fmax = 7
    elif band == "delta":
        fmin = 1
        fmax = 3
    
    if window == "early":
        tmin = 0
        tmax = 250
    elif window == "late":
        tmin = 256
        tmax = 504
    
    fmin_idx = np.argmin(np.abs(freqs-fmin))
    # assert freqs[fmin_idx] == fmin, "Check your freqs!"
    fmax_idx = np.argmin(np.abs(freqs-fmax))
    # assert freqs[fmax_idx] == fmax, "Check your freqs!"
    
    tmin_idx = np.argmin(np.abs(times-tmin))
    # assert times[tmin_idx] == tmin, "Check your times!"
    tmax_idx = np.argmin(np.abs(times-tmax))
    # assert times[tmax_idx] == tmax, "Check your times!"
    
    ch_idx = []
    for channel in ch:
        if channel in ch_locs:
            ch_idx.append(ch_locs.index(channel))
    for k, c in enumerate(conditions):
        context = ["s", "ns"]
        # sub_idx = scipy.io.loadmat(f"{arr_path}/idx_{c}.mat")["sub_idx"][0]-1 # make it 0-based again
        tf_df = pd.DataFrame(
            columns = [
            "sub",
            f"{m}_{context[0]}_err_min_corr_{band}_{window}",
            f"{m}_{context[1]}_err_min_corr_{band}_{window}",
        ]
        )
        
        tf_arr_c1= scipy.io.loadmat(f"{arr_path}/{m}_{c[0]}.mat")
        tf_arr_c2= scipy.io.loadmat(f"{arr_path}/{m}_{c[1]}.mat")
         
        sub_ids_c1 = tf_arr_c1['subjects'][0]
        sub_ids_c2 = tf_arr_c2['subjects'][0]
        
        tf_data_c1 = tf_arr_c1[f"{m}_{c[0]}"]
        tf_data_c2 = tf_arr_c2[f"{m}_{c[1]}"]
        
        assert tf_data_c1.shape[1:] == (64, 375, 59), f"Check your {m} data!"
        assert tf_data_c2.shape[1:] == (64, 375, 59), f"Check your {m} data!"
        
        # for sub_id in sub_idx:
        for sub_id in range(tf_data_c1.shape[0]):
            # sub_avg = np.mean(tf_data[sub_id, :, :, :], 0)
            sub_avg_c1 = tf_data_c1[sub_id, :, :, :]
            assert sub_avg_c1.shape == (64, 375, 59), f"Check your {m} data!"
            
            try:
                sub_to_match_c2 = list(sub_ids_c2).index(sub_ids_c1[sub_id])
                sub_avg_c2 = tf_data_c2[sub_to_match_c2, :, :, :]
                assert sub_avg_c2.shape == (64, 375, 59), f"Check your {m} data!"
                
                assert(sub_ids_c1[sub_id] == sub_ids_c2[sub_to_match_c2]), "Your IDs are messed up!"
                
                sub_avg_d1 = np.subtract(sub_avg_c1, sub_avg_c2)  # s err minus s corr
                assert sub_avg_d1.shape == (64, 375, 59), f"Check your {m} data!"
                
                ch_avg_d1 = np.mean(sub_avg_d1[ch_idx, :, :], 0)
                assert ch_avg_d1.shape == (375, 59), f"Check your {m} data!"
                
                time_avg_d1 = np.mean(ch_avg_d1[tmin_idx:tmax_idx+1, :], 0)
                assert len(time_avg_d1) == 59 and time_avg_d1.ndim == 1, f"Check your {m} data!"
                
                freq_avg_d1 = np.mean(time_avg_d1[fmin_idx:fmax_idx+1], 0)
            
                tf_df.loc[sub_id, "sub"] = sub_ids_c1[sub_id]
                tf_df.loc[sub_id, f"{m}_{context[k]}_err_min_corr_{band}_{window}"] = freq_avg_d1
            except:
                tf_df.loc[sub_id, "sub"] = np.nan
                tf_df.loc[sub_id, f"{m}_{context[k]}_err_min_corr_{band}_{window}"] = np.nan
            
        thrive_data = thrive_data.merge(tf_df, on="sub", how="left")

thrive_data = thrive_data.dropna(axis=1, how='all')#.to_csv("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/csv/thrive_TF_diff.csv", index=False)

thrive_data_soc = thrive_data[
    [i for i in thrive_data.columns if (i=="sub") or ("_s_" in i)]
]

thrive_data_soc.to_csv("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/csv/thrive_TF_diff_soc.csv", index=False)

thrive_data_nonsoc = thrive_data[
    [i for i in thrive_data.columns if (i=="sub") or ("_ns_" in i)]
]

thrive_data_nonsoc.to_csv("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/csv/thrive_TF_diff_nonsoc.csv", index=False)

In [64]:
# DIFFERENCE SCORES ICPS

thrive_data = pd.read_csv("/Users/fzaki001/Downloads/thrive/derivatives/processed/summary.csv")["sub"].to_frame()

# m = "TF"
conditions = [
    ["resp_s_i_0",
    "resp_s_i_1",],
    ["resp_ns_i_0",
    "resp_ns_i_1",],
            ]

# for m in measures:
    # for c in conditions:
        # for band in ["theta", "delta"]:
            # for window in ["early", "late"]:
for m in ["ICPS"]:
    band = 'theta'
    window = 'early'
    if band == "theta":
        fmin = 4
        fmax = 7
    elif band == "delta":
        fmin = 1
        fmax = 3
    
    if window == "early":
        tmin = 0
        tmax = 250
    elif window == "late":
        tmin = 256
        tmax = 504
    
    fmin_idx = np.argmin(np.abs(freqs-fmin))
    # assert freqs[fmin_idx] == fmin, "Check your freqs!"
    fmax_idx = np.argmin(np.abs(freqs-fmax))
    # assert freqs[fmax_idx] == fmax, "Check your freqs!"
    
    tmin_idx = np.argmin(np.abs(times-tmin))
    # assert times[tmin_idx] == tmin, "Check your times!"
    tmax_idx = np.argmin(np.abs(times-tmax))
    # assert times[tmax_idx] == tmax, "Check your times!"

    for cluster in ["OCC", "DLPFC"]:
        if cluster == "DLPFC":
            ch = ['4', '6', '36', '39']
        elif cluster == "OCC":
            ch = ['22', '24', '53', '55']
        for channel in ch:
            if channel in ch_locs:
                ch_idx.append(ch_locs.index(channel))
        for k, c in enumerate(conditions):
            context = ["s", "ns"]
            # sub_idx = scipy.io.loadmat(f"{arr_path}/idx_{c}.mat")["sub_idx"][0]-1 # make it 0-based again
            tf_df = pd.DataFrame(
                columns = [
                "sub",
                f"{m}_{cluster}_{context[0]}_err_min_corr_{band}_{window}",
                f"{m}_{cluster}_{context[1]}_err_min_corr_{band}_{window}",
            ]
            )
            
            tf_arr_c1= scipy.io.loadmat(f"{arr_path}/{m}_{c[0]}.mat")
            tf_arr_c2= scipy.io.loadmat(f"{arr_path}/{m}_{c[1]}.mat")
             
            sub_ids_c1 = tf_arr_c1['subjects'][0]
            sub_ids_c2 = tf_arr_c2['subjects'][0]
            
            tf_data_c1 = tf_arr_c1[f"{m}_{c[0]}"]
            tf_data_c2 = tf_arr_c2[f"{m}_{c[1]}"]
            
            assert tf_data_c1.shape[1:] == (64, 375, 59), f"Check your {m} data!"
            assert tf_data_c2.shape[1:] == (64, 375, 59), f"Check your {m} data!"
            
            # for sub_id in sub_idx:
            for sub_id in range(tf_data_c1.shape[0]):
                # sub_avg = np.mean(tf_data[sub_id, :, :, :], 0)
                sub_avg_c1 = tf_data_c1[sub_id, :, :, :]
                assert sub_avg_c1.shape == (64, 375, 59), f"Check your {m} data!"
                
                try:
                    sub_to_match_c2 = list(sub_ids_c2).index(sub_ids_c1[sub_id])
                    sub_avg_c2 = tf_data_c2[sub_to_match_c2, :, :, :]
                    assert sub_avg_c2.shape == (64, 375, 59), f"Check your {m} data!"
                    
                    assert(sub_ids_c1[sub_id] == sub_ids_c2[sub_to_match_c2]), "Your IDs are messed up!"
                    
                    sub_avg_d1 = np.subtract(sub_avg_c1, sub_avg_c2)  # s err minus s corr
                    assert sub_avg_d1.shape == (64, 375, 59), f"Check your {m} data!"
                    
                    ch_avg_d1 = np.mean(sub_avg_d1[ch_idx, :, :], 0)
                    assert ch_avg_d1.shape == (375, 59), f"Check your {m} data!"
                    
                    time_avg_d1 = np.mean(ch_avg_d1[tmin_idx:tmax_idx+1, :], 0)
                    assert len(time_avg_d1) == 59 and time_avg_d1.ndim == 1, f"Check your {m} data!"
                    
                    freq_avg_d1 = np.mean(time_avg_d1[fmin_idx:fmax_idx+1], 0)
                
                    tf_df.loc[sub_id, "sub"] = sub_ids_c1[sub_id]
                    tf_df.loc[sub_id, f"{m}_{cluster}_{context[k]}_err_min_corr_{band}_{window}"] = freq_avg_d1
                except:
                    tf_df.loc[sub_id, "sub"] = np.nan
                    tf_df.loc[sub_id, f"{m}_{cluster}_{context[k]}_err_min_corr_{band}_{window}"] = np.nan
                
            thrive_data = thrive_data.merge(tf_df, on="sub", how="left")

thrive_data = thrive_data.dropna(axis=1, how='all')#.to_csv("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/csv/thrive_TF_diff.csv", index=False)

thrive_data_soc = thrive_data[
    [i for i in thrive_data.columns if (i=="sub") or ("_s_" in i)]
]

thrive_data_soc.to_csv("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/csv/thrive_icps_diff_soc.csv", index=False)

thrive_data_nonsoc = thrive_data[
    [i for i in thrive_data.columns if (i=="sub") or ("_ns_" in i)]
]

thrive_data_nonsoc.to_csv("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/csv/thrive_icps_diff_nonsoc.csv", index=False)

In [62]:
thrive_data_soc = thrive_data[
    [i for i in thrive_data.columns if (i=="sub") or ("_s_" in i)]
]

thrive_data_soc.to_csv("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/csv/thrive_TF_diff_soc.csv", index=False)

thrive_data_nonsoc = thrive_data[
    [i for i in thrive_data.columns if (i=="sub") or ("_ns_" in i)]
]

thrive_data_nonsoc.to_csv("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/csv/thrive_TF_diff_nonsoc.csv", index=False)

In [51]:
thrive_data.columns = [
    "sub",
    "TF_s_err_min_corr", "TF_ns_err_min_corr", "TF_s_min_ns",
    "ITPS_s_err_min_corr", "ITPS_ns_err_min_corr", "ITPS_s_min_ns",
]
thrive_data.to_csv("/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/csv/thrive_diff.csv", index=False)

In [38]:
list(sub_ids_c3).index(sub_ids_c1[sub_id])

ValueError: 3000004 is not in list

In [337]:
csv_save_path = "/Users/fzaki001/thrive-theta-ddm/derivatives/preprocessed/csv"

thrive_tf_soc = thrive_data[
[i for i in thrive_data.columns if ("_s_" in i or "sub" in i)]
]

thrive_tf_nonsoc = thrive_data[
[i for i in thrive_data.columns if ("_ns_" in i or "sub" in i)]
]

thrive_tf_soc.to_csv(f"{csv_save_path}/thrive_{measures[0]}_soc.csv", index=False)
thrive_tf_nonsoc.to_csv(f"{csv_save_path}/thrive_{measures[0]}_nonsoc.csv", index=False)

In [322]:
df = thrive_tf_soc.copy()
df.head()

,sub,TF_resp_s_i_0_theta_early,TF_resp_s_i_0_theta_late,TF_resp_s_i_0_delta_early,TF_resp_s_i_0_delta_late,TF_resp_s_i_1_theta_early,TF_resp_s_i_1_theta_late,TF_resp_s_i_1_delta_early,TF_resp_s_i_1_delta_late,TF_resp_s_c_1_theta_early,...,ITPS_resp_s_i_0_delta_early,ITPS_resp_s_i_0_delta_late,ITPS_resp_s_i_1_theta_early,ITPS_resp_s_i_1_theta_late,ITPS_resp_s_i_1_delta_early,ITPS_resp_s_i_1_delta_late,ITPS_resp_s_c_1_theta_early,ITPS_resp_s_c_1_theta_late,ITPS_resp_s_c_1_delta_early,ITPS_resp_s_c_1_delta_late
0,3000001,0.959899,0.164399,2.114212,2.050642,-0.386809,-1.185442,0.671583,0.027389,-0.321639,...,0.215569,0.124596,0.067344,0.025451,0.047322,0.012075,0.071026,0.00479,0.075143,0.018349
1,3000002,2.304025,0.80378,1.108651,1.154355,1.318759,0.27168,0.823631,0.584494,0.863247,...,0.084003,0.072955,0.024469,0.010821,0.056578,0.037895,0.030313,0.021658,0.068549,0.052781
2,3000003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3000004,2.395475,1.028024,2.178461,1.697842,0.63161,-1.347773,0.562197,0.217295,0.984937,...,0.046656,-0.010204,0.046372,0.007188,0.084812,0.052478,0.056196,0.02311,0.080942,0.05094
4,3000005,2.315924,0.823528,1.036333,1.057553,0.438913,-1.48089,0.1569,-0.386233,-0.019179,...,0.056002,0.035601,0.014415,0.007576,0.029898,0.02975,0.027402,0.023467,0.050028,0.053385


In [268]:
ch = ['1', '2', '33', '34']


if band == "theta":
    fmin = 4
    fmax = 7
elif band == "delta":
    fmin = 1
    fmax = 3

if window == "early":
    tmin = 0
    tmax = 250
elif window == "late":
    tmin = 250
    tmax = 500

m = "TF"
c = "resp_s_i_0"

fmin_idx = np.argmin(np.abs(freqs-fmin))
# assert freqs[fmin_idx] == fmin, "Check your freqs!"
fmax_idx = np.argmin(np.abs(freqs-fmax))
# assert freqs[fmax_idx] == fmax, "Check your freqs!"

tmin_idx = np.argmin(np.abs(times-tmin))
# assert times[tmin_idx] == tmin, "Check your times!"
tmax_idx = np.argmin(np.abs(times-tmax))
# assert times[tmax_idx] == tmax, "Check your times!"

ch_idx = []
for channel in ch:
    if channel in ch_locs:
        ch_idx.append(ch_locs.index(channel))


# sub_idx = scipy.io.loadmat(f"{arr_path}/idx_{c}.mat")["sub_idx"][0]-1 # make it 0-based again
tf_df = pd.DataFrame(columns = ["sub", f"{m}_{c}_{band}_{window}"])
tf_arr = scipy.io.loadmat(f"{arr_path}/{m}_{c}.mat")
sub_ids = tf_arr['subjects'][0]
tf_data = tf_arr[f"{m}_{c}"]
assert tf_data.shape[1:] == (64, 375, 59), f"Check your {m} data!"

# for sub_id in sub_idx:
for sub_id in range(tf_data.shape[0]):
    # sub_avg = np.mean(tf_data[sub_id, :, :, :], 0)
    sub_avg = tf_data[sub_id, :, :, :]
    assert sub_avg.shape == (64, 375, 59), f"Check your {m} data!"
    
    ch_avg = np.mean(sub_avg[ch_idx, :, :], 0)
    assert ch_avg.shape == (375, 59), f"Check your {m} data!"
    
    time_avg = np.mean(ch_avg[tmin_idx:tmax_idx+1, :], 0)
    assert len(time_avg) == 59 and time_avg.ndim == 1, f"Check your {m} data!"
    freq_avg = np.mean(time_avg[fmin_idx:fmax_idx+1], 0)

    tf_df.loc[sub_id, "sub"] = sub_ids[sub_id]
    tf_df.loc[sub_id, f"{m}_{c}_{band}_{window}"] = freq_avg

In [257]:
tf_df = pd.DataFrame(columns = ["sub", f"c"])

In [265]:
len(time_avg) == 59 and time_avg.ndim == 1

True